In [24]:
import sys 
import os

# Add the parent directory to the path if it's not already there
if os.path.abspath('..') not in sys.path:
    sys.path.append(os.path.abspath('..'))

In [25]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
from ddql_optimal_execution import DDQL, MarketEnvironnement, MarketEnvironnementOld, Trainer, TWAP
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

In [27]:
QV = True  #next true, next false
Volume = False #next False, next false

In [28]:
env = MarketEnvironnement(initial_inventory=500, multi_episodes=True, QV=QV, Volume=Volume, data_path='../data/train')

In [29]:
agent = DDQL(state_size=env.state_size, initial_budget=env.initial_inventory, horizon=env.horizon)

Using cpu device


In [30]:
trainer = Trainer(agent, env, capacity=10000)

In [31]:
trainer.fill_exp_replay(max_steps=10000)

Filling experience replay buffer: 100%|██████████████████████████████████████████| 10000/10000 [00:12<00:00, 831.60it/s]


In [32]:
trainer.pretrain(max_steps=100, batch_size=128)

Pretraining agent: 100%|██████████████████████████████████████████████████████████████| 100/100 [00:03<00:00, 28.29it/s]


In [33]:
trainer.agent.greediness = 0.9

In [34]:
trainer.train(max_steps=1000, batch_size=128)

Training agent: 100%|███████████████████████████████████████████████████████████████| 1000/1000 [00:35<00:00, 28.54it/s]


In [35]:
# ── helpers ──────────────────────────────────────────────────────────────────
def run_evaluation(trainer, test_env, twap_cls, n_episodes=100):
    pnl_ddql, pnl_twap = [], []
    n_ep = min(len(test_env.historical_data_series), n_episodes)
    random_ep = np.random.choice(np.arange(n_ep), size=n_ep, replace=True)

    for ep in random_ep:
        # TWAP
        test_env.swap_episode(ep)
        twap = twap_cls(initial_budget=test_env.initial_inventory, horizon=test_env.horizon)
        while not test_env.done:
            action = twap(test_env.state.copy())
            _ = test_env.step(action)
        pnl_twap.append(np.sum(test_env.pnl_for_episode))
        test_env.reset()

        # DDQL
        test_env.swap_episode(ep)
        while not test_env.done:
            action = trainer.agent(test_env.state.copy())
            _ = test_env.step(action)
        pnl_ddql.append(np.sum(test_env.pnl_for_episode))
        test_env.reset()

    return np.array(pnl_ddql), np.array(pnl_twap)


def print_comparison(label, pnl_ddql, pnl_twap):
    delta = pnl_ddql - pnl_twap
    glr   = delta[delta > 0].mean() / abs(delta[delta < 0].mean()) if (delta < 0).any() else float('inf')
    print(f"\n{'─'*45}")
    print(f"  {label}")
    print(f"{'─'*45}")
    print(f"  Mean   ΔPnL  : {delta.mean():.4f}")
    print(f"  Median ΔPnL  : {np.median(delta):.4f}")
    print(f"  Std    ΔPnL  : {delta.std():.4f}")
    print(f"  Win Prob      : {np.bincount(delta > 0)[1] / len(delta):.2%}")
    print(f"  GLR           : {glr:.4f}")
    print(f"{'─'*45}")


# ── build & train OLD env (original reward, lambda_risk=0) ───────────────────
print("Training OLD environment (original reward)...")
env_old = MarketEnvironnementOld(
    initial_inventory=500, multi_episodes=True,
    QV=True, Volume=False, data_path='../data/train'
)
agent_old  = DDQL(state_size=env_old.state_size, initial_budget=env_old.initial_inventory, horizon=env_old.horizon)
trainer_old = Trainer(agent_old, env_old, capacity=10000)
trainer_old.fill_exp_replay(max_steps=10000)
trainer_old.pretrain(max_steps=100, batch_size=128)
trainer_old.agent.greediness = 0.9
trainer_old.train(max_steps=1000, batch_size=128)

test_env_old = MarketEnvironnementOld(
    initial_inventory=500, multi_episodes=True,
    QV=True, Volume=False, data_path='../data/test'
)
pnl_ddql_old, pnl_twap_old = run_evaluation(trainer_old, test_env_old, TWAP)
print_comparison("OLD reward (lambda_risk = 0)", pnl_ddql_old, pnl_twap_old)


# ── build & train NEW env (Method B reward, lambda_risk=0.005) ───────────────
print("\nTraining NEW environment (Method B reward)...")
env_new = MarketEnvironnement(
    initial_inventory=500, multi_episodes=True,
    QV=True, Volume=False, data_path='../data/train',
    lambda_risk=0.15                              
)
agent_new   = DDQL(state_size=env_new.state_size, initial_budget=env_new.initial_inventory, horizon=env_new.horizon)
trainer_new = Trainer(agent_new, env_new, capacity=10000)
trainer_new.fill_exp_replay(max_steps=10000)
trainer_new.pretrain(max_steps=100, batch_size=128)
trainer_new.agent.greediness = 0.9
trainer_new.train(max_steps=1000, batch_size=128)

test_env_new = MarketEnvironnement(
    initial_inventory=500, multi_episodes=True,
    QV=True, Volume=False, data_path='../data/test',
    lambda_risk=0.15
)
pnl_ddql_new, pnl_twap_new = run_evaluation(trainer_new, test_env_new, TWAP)
print_comparison("NEW reward (Method B, lambda_risk=0.15)", pnl_ddql_new, pnl_twap_new)

Training OLD environment (original reward)...
Using cpu device


Training agent: 100%|███████████████████████████████████████████████████████████████| 1000/1000 [00:34<00:00, 29.02it/s]



─────────────────────────────────────────────
  OLD reward (lambda_risk = 0)
─────────────────────────────────────────────
  Mean   ΔPnL  : 17297.2036
  Median ΔPnL  : 10596.0947
  Std    ΔPnL  : 57310.4231
  Win Prob      : 61.00%
  GLR           : 1.4583
─────────────────────────────────────────────

Training NEW environment (Method B reward)...
Using cpu device


Filling experience replay buffer: 10001it [00:12, 799.71it/s]                                                           
Training agent: 100%|███████████████████████████████████████████████████████████████| 1000/1000 [00:34<00:00, 28.90it/s]



─────────────────────────────────────────────
  NEW reward (Method B, lambda_risk=0.15)
─────────────────────────────────────────────
  Mean   ΔPnL  : 5621.2533
  Median ΔPnL  : 3950.8538
  Std    ΔPnL  : 82514.8856
  Win Prob      : 51.00%
  GLR           : 1.1380
─────────────────────────────────────────────


In [36]:
import pandas as pd

lambda_grid = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3]
results = []

for lam in lambda_grid:
    print(f"\n── lambda_risk = {lam} ──────────────────────────")

    # Train
    env_lam = MarketEnvironnement(
        initial_inventory=500, multi_episodes=True,
        QV=True, Volume=False,
        data_path='../data/train',
        lambda_risk=lam
    )
    agent_lam   = DDQL(state_size=env_lam.state_size, initial_budget=env_lam.initial_inventory, horizon=env_lam.horizon)
    trainer_lam = Trainer(agent_lam, env_lam, capacity=10000)
    trainer_lam.fill_exp_replay(max_steps=10000)
    trainer_lam.pretrain(max_steps=100, batch_size=128)
    trainer_lam.agent.greediness = 0.9
    trainer_lam.train(max_steps=1000, batch_size=128)

    # Evaluate
    test_env_lam = MarketEnvironnement(
        initial_inventory=500, multi_episodes=True,
        QV=True, Volume=False,
        data_path='../data/test',
        lambda_risk=lam                    # pass same lambda at test time
    )
    pnl_ddql_lam, pnl_twap_lam = run_evaluation(trainer_lam, test_env_lam, TWAP)

    delta = pnl_ddql_lam - pnl_twap_lam
    glr   = delta[delta > 0].mean() / abs(delta[delta < 0].mean()) if (delta < 0).any() else float('inf')

    row = {
        "lambda_risk"  : lam,
        "mean_delta"   : delta.mean(),
        "median_delta" : np.median(delta),
        "std_delta"    : delta.std(),
        "win_prob"     : np.bincount(delta > 0)[1] / len(delta),
        "glr"          : glr,
        "pnl_variance": np.var(pnl_ddql_lam),   # lower = less risky
        "sharpe_proxy": delta.mean() / delta.std(),  # mean outperformance per unit of variability
    }
    results.append(row)
    print(f"  Mean ΔPnL={row['mean_delta']:.4f}  Win={row['win_prob']:.2%}  GLR={row['glr']:.4f}  Std={row['std_delta']:.2f}  Sharpe={row['sharpe_proxy']:.4f}  PnL Var={row['pnl_variance']:.2f}")

# ── summary table ─────────────────────────────────────────────────────────────
results_df = pd.DataFrame(results).set_index("lambda_risk")
print("\n\n════ LAMBDA GRID SEARCH SUMMARY ════")
print(results_df.to_string(float_format="{:.4f}".format))

# pick best by win probability
best_lam = results_df["win_prob"].idxmax()
print(f"\n→ Best lambda_risk by win prob: {best_lam}")


── lambda_risk = 0.05 ──────────────────────────
Using cpu device


Filling experience replay buffer: 10001it [00:12, 812.16it/s]                                                           
Training agent: 100%|███████████████████████████████████████████████████████████████| 1000/1000 [00:34<00:00, 28.70it/s]


  Mean ΔPnL=10478.3478  Win=46.00%  GLR=1.7647  Std=66736.54  Sharpe=0.1570  PnL Var=35334771161.62

── lambda_risk = 0.1 ──────────────────────────
Using cpu device


Filling experience replay buffer: 10001it [00:12, 793.47it/s]                                                           
Training agent: 100%|███████████████████████████████████████████████████████████████| 1000/1000 [00:34<00:00, 29.18it/s]


  Mean ΔPnL=5587.2534  Win=46.00%  GLR=1.4194  Std=73815.24  Sharpe=0.0757  PnL Var=27857637713.41

── lambda_risk = 0.15 ──────────────────────────
Using cpu device


...

  Mean ΔPnL=8423.9405  Win=51.00%  GLR=1.4368  Std=54218.61  Sharpe=0.1554  PnL Var=23161514151.13

── lambda_risk = 0.25 ──────────────────────────
Using cpu device


Filling experience replay buffer: 10001it [00:12, 824.12it/s]                                                           
Training agent: 100%|███████████████████████████████████████████████████████████████| 1000/1000 [00:34<00:00, 29.03it/s]


  Mean ΔPnL=-3160.3616  Win=49.00%  GLR=0.9679  Std=108113.58  Sharpe=-0.0292  PnL Var=12001147669.43

── lambda_risk = 0.3 ──────────────────────────
Using cpu device


Filling experience replay buffer: 10002it [00:12, 798.29it/s]                                                           
Training agent: 100%|███████████████████████████████████████████████████████████████| 1000/1000 [00:34<00:00, 28.76it/s]


  Mean ΔPnL=12618.6420  Win=60.00%  GLR=1.0502  Std=68664.40  Sharpe=0.1838  PnL Var=20501020909.08


════ LAMBDA GRID SEARCH SUMMARY ════
             mean_delta  median_delta   std_delta  win_prob    glr     pnl_variance  sharpe_proxy
lambda_risk                                                                                      
0.05         10478.3478    -7318.1547  66736.5404    0.4600 1.7647 35334771161.6196        0.1570
0.10          5587.2534   -11252.3313  73815.2443    0.4600 1.4194 27857637713.4097        0.0757
0.15          -665.6354    -2105.0418  51462.7366    0.4700 1.0908 20768404620.6897       -0.0129
0.20          8423.9405     3541.7288  54218.6082    0.5100 1.4368 23161514151.1312        0.1554
0.25         -3160.3616    -3249.9741 108113.5793    0.4900 0.9679 12001147669.4293       -0.0292
0.30         12618.6420    23267.2393  68664.3990    0.6000 1.0502 20501020909.0796        0.1838

→ Best lambda_risk by win prob: 0.3


In [39]:
pnl_twap = []
pnl_ddql = []

pnl_ddql, pnl_twap = np.array(pnl_ddql), np.array(pnl_twap)

In [40]:
pnl_ddql_sum = pnl_ddql.sum(axis=1)
pnl_twap_sum = pnl_twap.sum(axis=1)

delta_pnl = (pnl_ddql_sum - pnl_twap_sum)/ pnl_twap_sum



GLR = - delta_pnl[delta_pnl > 0].mean()/  delta_pnl[delta_pnl < 0].mean()

prob_win = np.bincount(delta_pnl > 0)[1] / len(delta_pnl)


AxisError: axis 1 is out of bounds for array of dimension 1

In [ ]:
for QV in [True, ]:
    for Volume in [False, ]:
        env = MarketEnvironnement(initial_inventory=500, multi_episodes=True, QV=QV, Volume=Volume, data_path='../data/train', quadratic_penalty_coefficient=0)
        agent = DDQL(state_size=env.state_size, initial_budget=env.initial_inventory, horizon=env.horizon, quadratic_penalty_coefficient=0)
        trainer = Trainer(agent, env, capacity=10000)
        trainer.fill_exp_replay(max_steps=10000)
        trainer.pretrain(max_steps=100, batch_size=128)
        trainer.agent.greediness = 0.9
        trainer.train(max_steps=1000, batch_size=128)



        #test the agent
        for data_path in ['train', 'test']:
            test_env = MarketEnvironnement(initial_inventory=500, multi_episodes=True, QV=QV, Volume=Volume, data_path=f'../data/{data_path}', quadratic_penalty_coefficient=0)

            twap = TWAP(initial_budget=test_env.initial_inventory, horizon=test_env.horizon)

            pnl_twap = []
            pnl_ddql = []


            n_episodes = min(len(test_env.historical_data_series), 100)

            random_ep = np.random.choice(np.arange(n_episodes), size=n_episodes, replace=True)

            for ep in random_ep:
                test_env.swap_episode(ep)
                _pnl_twap = [0]
                while not test_env.done:
                    current_state = test_env.state.copy()
                    action = twap(current_state)
                    _ = test_env.step(action)
                    
                pnl_twap.append(test_env.pnl_for_episode + [test_env.state['Price']*test_env.state['inventory'] - test_env.quadratic_penalty_coefficient*(test_env.state['inventory']/test_env.initial_inventory)**2 / test_env.horizon])

                test_env.reset()
                
                _pnl_ddql = [0]
                while not test_env.done:
                    current_state = test_env.state.copy()
                    action = trainer.agent(current_state)
                    _ = test_env.step(action)
                pnl_ddql.append(test_env.pnl_for_episode + [test_env.state['Price']*test_env.state['inventory'] - test_env.quadratic_penalty_coefficient*(test_env.state['inventory']/test_env.initial_inventory)**2 / test_env.horizon])

            pnl_ddql, pnl_twap = np.array(pnl_ddql), np.array(pnl_twap)

            pnl_ddql_sum = pnl_ddql.sum(axis=1)
            pnl_twap_sum = pnl_twap.sum(axis=1)

            delta_pnl = (pnl_ddql_sum - pnl_twap_sum)/ pnl_twap_sum

            GLR = - delta_pnl[delta_pnl > 0].mean()/  delta_pnl[delta_pnl < 0].mean()

            prob_win = np.bincount(delta_pnl > 0)[1] / len(delta_pnl)
            sns.histplot(delta_pnl, kde=True, stat='frequency')
            #change font$

            title = "\n".join([r'$\Delta$P&L', 'TIPTr' + 'V'*Volume+'QV'*QV])
            plt.title(title, fontdict={'fontfamily': 'times new roman'}, x=0.5, y=1, fontsize=20)
            x, y = plt.xlim()[0]*.9, plt.ylim()[1] * .7
            plt.text(x=x, y=y, s="\n".join([r'$\mathbb{P}(\Delta{P&L}'+r'>0 ) = {:.2f}$'.format(prob_win) , 
                        r'$Mean = {:.2f}$'.format(delta_pnl.mean()), r'$Std = {:.2f}$'.format(delta_pnl.std()), r'$Median = {:.2f}$'.format(np.median(delta_pnl)), r' $GLR = {:.2f}$'.format(GLR)]), fontsize=12, fontdict={'fontfamily': 'times new roman'})

            #save with ransparent background

            plt.savefig(title.replace("\n", "_").replace("$", "").replace("\\", "").replace("&", "n") +f'_{data_path}.png', dpi=300, bbox_inches='tight',transparent=True)
            plt.show()

In [ ]:
sns.histplot(pnl_twap.sum(axis=1), label='ddql', kde=True, stat='density', color='tab:red', alpha=.5)
sns.histplot(pnl_ddql.sum(axis=1), label='twap', kde=True, stat='density', color='tab:blue', alpha=.5)
plt.legend()
plt.title('Distribution of PnL')
plt.xlabel('PnL')

In [ ]:
np.mean(pnl_twap.sum(axis=1))-np.mean(pnl_ddql.sum(axis=1))

In [ ]:
grid = {
    "QV": [True, False],
    "Volume": [False],

}

data = dict()

for QV in grid["QV"]:
    for Volume in grid["Volume"]:
        env = MarketEnvironnement(initial_inventory=100, multi_episodes=True, QV=QV, Volume=Volume, data_path='../data/train')
        agent = DDQL(state_size=env.state_size, initial_budget=env.initial_inventory, horizon=env.horizon)
        trainer = Trainer(agent, env, capacity=10000)
        trainer.fill_exp_replay(max_steps=10000)
        trainer.pretrain(max_steps=1000, batch_size=128)
        trainer.train(max_steps=10000, batch_size=128)
        env.reset()
        twap = TWAP(initial_budget=env.initial_inventory, horizon=env.horizon)
        pnl_twap = []
        pnl_ddql = []
        random_ep = np.random.choice(np.arange(len(env.historical_data_series)), 100)
        for ep in random_ep:
            env.swap_episode(ep)
            _pnl_twap = [0]
            while not env.done:
                current_state = env.state.copy()
                action = twap(current_state)
                _ = env.step(action)
                _pnl_twap.append(env.state['Price']*action)
            pnl_twap.append(_pnl_twap)
            env.reset()
            _pnl_ddql = [0]
            while not env.done:
                current_state = env.state.copy()
                action = trainer.agent(current_state)
                _ = env.step(action)
                _pnl_ddql.append(env.state['Price']*action)
            pnl_ddql.append(_pnl_ddql)

        pnl_ddql, pnl_twap = np.array(pnl_ddql), np.array(pnl_twap)
        data[(QV, Volume)] = (pnl_ddql, pnl_twap)



In [ ]:
import pickle

with open('data.pkl', 'wb') as f:
    pickle.dump(data, f)

In [ ]:
sns.set_palette('pastel')
sns.set_style('whitegrid')

In [ ]:
for i, (QV, Volume) in enumerate(data):
    pnl_ddql, pnl_twap = data[(QV, Volume)]
    
    confidence_level_2 = .99
    plt.plot(pnl_ddql.mean(axis=0), label='ddql', color='tab:blue')
    plt.fill_between(range(len(pnl_ddql.mean(axis=0))), pnl_ddql.mean(axis=0)-confidence_level_2*pnl_ddql.std(axis=0)/np.sqrt(len(pnl_ddql)), pnl_ddql.mean(axis=0)+confidence_level_2*pnl_ddql.std(axis=0)/np.sqrt(len(pnl_ddql)), alpha=.1,   color='tab:blue')

    plt.plot(pnl_twap.mean(axis=0), label='twap', color='tab:red')
    plt.fill_between(range(len(pnl_twap.mean(axis=0))), pnl_twap.mean(axis=0)-confidence_level_2*pnl_twap.std(axis=0)/np.sqrt(len(pnl_twap)), pnl_twap.mean(axis=0)+confidence_level_2*pnl_twap.std(axis=0)/np.sqrt(len(pnl_twap)), alpha=.1, color='tab:red')
    plt.legend()
    plt.xticks(
        #rotation=45,
        ticks=range(5), labels=[f"{i}" for i in range(5)]
    )
    plt.title(f'Profits with normalized prices (with 99% confidence interval)\n QV={QV}, Volume={Volume}')
    
    plt.ylabel('PnL')
    plt.savefig(f'pnl_QV_{QV}_Volume_{Volume}.png')
    plt.show()

In [ ]:
for key in data:
    print(key, np.mean(data[key][0].sum(axis=1))-np.mean(data[key][1].sum(axis=1)))
    #plt.hist(data[key][0].sum(axis=1), label='ddql', alpha=.5, color='tab:blue')
    #plt.axvline(np.mean(data[key][0].sum(axis=1)), color='tab:blue')
    #plt.hist(data[key][1].sum(axis=1), label='twap', alpha=.5, color='tab:red')
    #plt.axvline(np.mean(data[key][1].sum(axis=1)), color='tab:red')

    pnl_twap, pnl_ddql = data[key]

    sns.histplot(pnl_twap.sum(axis=1), label='ddql', kde=True, stat='probability', color='tab:red', alpha=.5)
    sns.histplot(pnl_ddql.sum(axis=1), label='twap', kde=True, stat='probability', color='tab:blue', alpha=.5)
    #plt.legend()
    #plt.title('')
    plt.xlabel('PnL')

    plt.legend()
    plt.title(f'Distribution of PnL \nQV={key[0]}, Volume={key[1]}, Pnl Spread = {np.mean(data[key][0].sum(axis=1))-np.mean(data[key][1].sum(axis=1)):.2f}')
    plt.savefig(f'../figs/QV_{key[0]}_Volume_{key[1]}_hists.png')
    plt.show()


In [ ]:
#for i in range(100):
#    fake_data(start="2022-01-01 11:00:01", end="2022-01-01 12:00:00").to_csv(f"../data/fake_data_{i}.csv")
#    fake_data(start="2022-01-01 12:00:01", end="2022-01-01 13:00:00").to_csv(f"../data/fake_data_{i+100}.csv")
#    fake_data(start="2022-01-01 13:00:01", end="2022-01-01 14:00:00").to_csv(f"../data/fake_data_{i+200}.csv")AA

In [ ]:
# healthcare data with old reward function

import numpy as np
import matplotlib.pyplot as plt

# 1. Initialize the original paper's environment (Risk-Neutral baseline)
print("Initializing the original risk-neutral paper environment...")
env_old = MarketEnvironnementOld(
    initial_inventory=500, 
    multi_episodes=True, 
    QV=True, 
    Volume=False, 
    data_path='../data/train'
)

# 2. Build the Agent and Trainer structures
agent_old = DDQL(
    state_size=env_old.state_size, 
    initial_budget=env_old.initial_inventory, 
    horizon=env_old.horizon
)
trainer_old = Trainer(agent_old, env_old, capacity=10000)

# 3. Replay Buffer & Pretraining steps
print("Filling experience replay buffer...")
trainer_old.fill_exp_replay(max_steps=10000)

print("Pretraining the baseline agent...")
trainer_old.pretrain(max_steps=100, batch_size=128)

# 4. Train the risk-neutral agent
trainer_old.agent.greediness = 0.9
print("Training the baseline agent (1000 steps)...")
trainer_old.train(max_steps=1000, batch_size=128)
print("Baseline model training complete!\n")

# 5. Evaluate the baseline on the unseen Test data
print("Evaluating the risk-neutral paper baseline on test data...")
test_env_old = MarketEnvironnementOld(
    initial_inventory=500, 
    multi_episodes=True, 
    QV=True, 
    Volume=False, 
    data_path='../data/test'
)

pnl_twap_old = []
pnl_ddql_old = []
n_episodes = len(test_env_old.historical_data_series)

for ep in range(n_episodes):
    # Test TWAP Baseline
    test_env_old.swap_episode(ep)
    twap = TWAP(initial_budget=test_env_old.initial_inventory, horizon=test_env_old.horizon)
    
    while not test_env_old.done:
        current_state = test_env_old.state.copy()
        action = twap(current_state)
        _ = test_env_old.step(action)
        
    # Directly grab the total accumulated P&L for the entire episode
    pnl_twap_old.append(np.sum(test_env_old.pnl_for_episode))
    test_env_old.reset()
    
    # Test DDQL Risk-Neutral Agent
    test_env_old.swap_episode(ep)
    while not test_env_old.done:
        current_state = test_env_old.state.copy()
        action = trainer_old.agent(current_state) 
        _ = test_env_old.step(action)
        
    pnl_ddql_old.append(np.sum(test_env_old.pnl_for_episode))
    test_env_old.reset()

# 6. Compute and display summary stats cleanly
sum_ddql = np.array(pnl_ddql_old)
sum_twap = np.array(pnl_twap_old)
delta_pnl_old = sum_ddql - sum_twap

print(f"--- Original Paper Baseline Results (Healthcare Data) ---")
print(f"Mean Out-performance vs TWAP: {np.mean(delta_pnl_old):.4f}")
print(f"Median Out-performance vs TWAP: {np.median(delta_pnl_old):.4f}")
print(f"Win Probability P(ΔPnL > 0): {np.sum(delta_pnl_old > 0) / len(delta_pnl_old) * 100:.2f}%\n")